# Scan del guadagno PMT: log(Q_A/Q_B) vs X

Script NUOVO e autonomo (non tocca `Signal_maker.ipynb` ne' `WC_analisys_tool.ipynb`).
Fisica del PMT/cavo presa da `Signal_maker.ipynb` (H14603-103, cavo 6 m); estrazione
carica (baseline + gate dinamico) presa da `WC_analisys_tool.ipynb`, riscritte qui in
forma minimale e indipendente.

**Idea**: per ogni evento, la parte stocastica che NON dipende dal guadagno (accettazione
QE dei fotoni, jitter TTS, fluttuazione SPE_RES per fotone) viene generata **una sola
volta**; per ogni valore di guadagno si scala solo l'ampiezza dell'impulso, si rifà la
quantizzazione ADC (con rumore elettronico indipendente, come nella realta') e si
ricalcola la carica col gate dinamico. Cosi' le curve a guadagni diversi condividono la
stessa realizzazione fotonica per evento -> confronto pulito, isola l'effetto del guadagno.

In [ ]:
import os, re, glob
import numpy as np
import uproot
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ============================================================
# CONFIG
# ============================================================
DATA_DIR   = "/Users/benussi/Testbeam2026_WC_unified/data"
OUT_DIR    = "/Users/benussi/Testbeam2026_WC_unified/Analysis_script/output_gain_scan"
os.makedirs(OUT_DIR, exist_ok=True)

GEO_SEL = "single"
SRC_SEL = "e_tb"
CH_A, CH_B = 0, 1                    # canali PMT per la geometria 'single'

GAIN_LIST = np.array([2.0e5, 4.0e5, 6.0e5, 8.0e5, 1.0e6])   # scan 2e5 -> 1e6
EVENTS_PER_FILE = 500                 # cap eventi per posizione X (None = tutti)
RNG_SEED = 12345

# ============================================================
# DIGITIZER / PMT / CAVO — stessa fisica di Signal_maker.ipynb (G escluso, e' lo scan)
# ============================================================
SAMPLE_RATE_GS = 2.5
DT_NS          = 1.0 / SAMPLE_RATE_GS        # 0.4 ns/campione
RECORD_LENGTH  = 1024
ADC_BITS       = 12
V_PP_MV        = 1000.0
BASELINE_ADC   = 3800
NOISE_RMS_ADC  = 3.5
TRIGGER_OFFSET_NS = 150.0
LSB = V_PP_MV / (2**ADC_BITS)

R            = 50.0
e_charge     = 1.602e-19
tr_pmt       = 0.6
TTS_FWHM_NS  = 0.3
TTS_SIGMA_NS = TTS_FWHM_NS / 2.355
SPE_RES      = 0.40

CABLE_LENGTH_M      = 6.0             # confermato: cavo 6 m (era 10 m)
ATTENUATION_DB      = (15.0/100.0) * CABLE_LENGTH_M
TRANSMISSION_FACTOR = 10.0 ** (-ATTENUATION_DB/20.0)
tr_cable            = 0.14 * CABLE_LENGTH_M
tr_total             = np.sqrt(tr_pmt**2 + tr_cable**2)
sigma_total          = tr_total / 2.22

def amp_spe_mv(gain):
    """Ampiezza SPE (mV, attenuata dal cavo) per un dato guadagno G."""
    ideal = -(e_charge * gain * R) / (sigma_total * 1e-9 * np.sqrt(2*np.pi)) * 1e3
    return ideal * TRANSMISSION_FACTOR

def get_quantum_efficiency(wavelengths_nm):
    """QE curve Hamamatsu H14603-103 (bialkali) — identica a Signal_maker."""
    wl = np.array([200.,250.,300.,350.,380.,400.,420.,450.,500.,550.,600.,650.,700.])
    qe = np.array([0.18,0.25,0.30,0.34,0.35,0.34,0.32,0.28,0.18,0.08,0.03,0.005,0.00])
    return np.interp(wavelengths_nm, wl, qe, left=0., right=0.)

time_axis = np.arange(0, RECORD_LENGTH*DT_NS, DT_NS)

def mv_to_adc(voltage_mv, rng):
    adc = BASELINE_ADC + (voltage_mv / LSB)
    adc = adc + rng.normal(0, NOISE_RMS_ADC, len(adc))
    return np.clip(np.round(adc), 0, (2**ADC_BITS)-1)

# ============================================================
# ESTRAZIONE CARICA — stessa logica (gate dinamico + piedistallo) di WC_analisys_tool.ipynb
# ============================================================
PMT_BL_LEFT_NS  = (0.0, 100.0)
PMT_BL_RIGHT_NS = (300.0, 400.0)
PMT_SIG_WIN_NS  = (100.0, 240.0)
Q_GATE_THR      = -2
Q_GATE_LEAD_NS  = 10
ADC_GATE_START_NS = 10.0
PULSE_WINDOW_NS   = 50.0
GATE_PED_NS       = (250.0, 260.0)   # finestra piedistallo (fissa, post-segnale) — stessa per MC
Q_PED_NSIGMA      = 3.0              # soglia selezione: Q > media_ped + N*sigma_ped
np_trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

def ns_to_samples(ns_value, dt_ns, min_val=1):
    return max(min_val, int(round(ns_value/dt_ns)))

def leading_edge_crossing(wf, thr, lead_samples=None):
    wf = np.asarray(wf)
    if len(wf) == 0: return None
    i_min = int(np.argmin(wf))
    if wf[i_min] > thr: return None
    cond = (wf[:-1] > thr) & (wf[1:] <= thr)
    idx = np.where(cond)[0]
    idx = idx[idx <= i_min]
    if lead_samples is not None:
        idx = idx[idx >= i_min - int(lead_samples)]
    if len(idx) == 0: return None
    i = int(idx[0]) if lead_samples is not None else int(idx[-1])
    y1, y2 = wf[i], wf[i+1]
    return float(i + (thr - y1)/(y2 - y1))

def fit_gauss(data):
    """Fit gaussiano robusto (mediana + MAD per l'inizializzazione) — stessa versione
    corretta usata in WC_analisys_tool.ipynb, copiata qui (script autonomo)."""
    data = np.asarray(data, dtype=float)
    n = len(data)
    if n < 2:
        raise RuntimeError("dati insufficienti")
    lo, hi = float(np.min(data)), float(np.max(data))
    if hi <= lo:
        raise RuntimeError("zero variance")
    mu_init = np.median(data)
    sig_init = np.median(np.abs(data - mu_init)) * 1.4826
    if sig_init == 0:
        sig_init = np.std(data)
    if sig_init == 0:
        raise RuntimeError("zero variance")
    fit_lo, fit_hi = mu_init - 5*sig_init, mu_init + 5*sig_init
    mask = (data >= fit_lo) & (data <= fit_hi)
    if mask.sum() < 5:
        fit_lo, fit_hi = lo, hi
    hist, bins = np.histogram(data, bins=50, range=(fit_lo, fit_hi))
    x = 0.5 * (bins[1:] + bins[:-1])
    popt, pcov = curve_fit(lambda x, a, m, s: a*np.exp(-(x-m)**2/(2*s**2)),
                           x, hist, p0=[n, mu_init, sig_init], maxfev=5000)
    return popt, np.sqrt(np.diag(pcov))

def ped_fit_thr(ped):
    """Media/sigma del piedistallo (fit gaussiano) e soglia a Q_PED_NSIGMA sigma."""
    if len(ped) < 10:
        return np.nan, np.nan, np.nan
    try:
        (_, mu, sg), _ = fit_gauss(np.asarray(ped, dtype=float))
        return mu, abs(sg), mu + Q_PED_NSIGMA*abs(sg)
    except Exception:
        return np.nan, np.nan, np.nan

_roi0 = max(0, int(round(PMT_SIG_WIN_NS[0]/DT_NS)))
_roi1 = max(0, int(round(PMT_SIG_WIN_NS[1]/DT_NS)))
_gshift = ns_to_samples(ADC_GATE_START_NS, DT_NS, min_val=0)
_gwin   = ns_to_samples(PULSE_WINDOW_NS, DT_NS)
_lead_q = ns_to_samples(Q_GATE_LEAD_NS, DT_NS, min_val=1)
_pp0 = max(0, int(round(GATE_PED_NS[0]/DT_NS))); _pp1 = max(_pp0+1, int(round(GATE_PED_NS[1]/DT_NS)))

def baseline_subtract(wf):
    t = np.arange(len(wf), dtype=float) * DT_NS
    l0 = max(0, int(round(PMT_BL_LEFT_NS[0]/DT_NS)));  l1 = max(l0+1, int(round(PMT_BL_LEFT_NS[1]/DT_NS)))
    r0 = max(0, int(round(PMT_BL_RIGHT_NS[0]/DT_NS))); r1 = max(r0+1, int(round(PMT_BL_RIGHT_NS[1]/DT_NS)))
    med_l = float(np.median(wf[l0:l1])); med_r = float(np.median(wf[r0:r1]))
    tc_l = 0.5*(PMT_BL_LEFT_NS[0]+PMT_BL_LEFT_NS[1]); tc_r = 0.5*(PMT_BL_RIGHT_NS[0]+PMT_BL_RIGHT_NS[1])
    slope = (med_r - med_l) / (tc_r - tc_l)
    baseline = med_l + slope*(t - tc_l)
    return wf - baseline

def extract_charge_and_ped(wf_adc):
    """Baseline-sub + gate dinamico -> (carica o np.nan, piedistallo [250,260]ns sempre)."""
    sig = baseline_subtract(wf_adc.astype(float))
    ped = float(-np_trapz(sig[_pp0:_pp1]))
    seg = sig[_roi0:_roi1]
    t_cross = leading_edge_crossing(seg, Q_GATE_THR, lead_samples=_lead_q)
    if t_cross is None:
        return np.nan, ped
    g0 = max(0, int(t_cross + _roi0) - _gshift)
    g1 = g0 + _gwin
    return float(-np_trapz(sig[g0:g1])), ped

print(f"Config: GEO={GEO_SEL} SRC={SRC_SEL}  guadagni={GAIN_LIST}")
print(f"amp_SPE(G) attenuata: " + ", ".join(f"G={g:.1e}->{amp_spe_mv(g):.2f}mV" for g in GAIN_LIST))
print(f"Selezione: coincidenza A&B (Q>0) + soglia piedistallo Q_PED_NSIGMA={Q_PED_NSIGMA} per canale (come WC_analisys_tool)")

In [ ]:
# ============================================================
# LETTURA ROOT + DIGITIZZAZIONE MULTI-GUADAGNO + ESTRAZIONE CARICA
# ============================================================
FILE_RE = re.compile(rf'sim_{GEO_SEL}_{SRC_SEL}_X(m?\d+)mm_Y(m?\d+)mm_Th(\d+)deg\.root$')

def parse_signed(s):
    return -float(s[1:]) if s.startswith('m') else float(s)

root_files = sorted(glob.glob(os.path.join(DATA_DIR, f"sim_{GEO_SEL}_{SRC_SEL}_X*.root")))
print(f"trovati {len(root_files)} file ROOT")

rng = np.random.default_rng(RNG_SEED)

# qa_qb_pairs[gain][x_cm] = lista di (Q_A, Q_B) per evento in coincidenza A&B (Q>0)
# ped_a[gain][x_cm] / ped_b[gain][x_cm] = piedistallo per canale, raccolto ogni volta
#   che QUEL canale trova un fronte (indipendentemente dall'altro canale) — stessa
#   logica di _pa_calo/_pb_calo in WC_analisys_tool.
qa_qb_pairs = {float(g): {} for g in GAIN_LIST}
ped_a       = {float(g): {} for g in GAIN_LIST}
ped_b       = {float(g): {} for g in GAIN_LIST}

for fpath in root_files:
    bn = os.path.basename(fpath)
    m = FILE_RE.search(bn)
    if not m:
        continue
    x_mm = parse_signed(m.group(1)); y_mm = parse_signed(m.group(2)); theta = float(m.group(3))
    if y_mm != 0:            # solo Y=0 per lo scan (come da configurazione richiesta)
        continue
    if theta != 0.0:         # solo run e_tb standard (Th0deg); esclude scan fine separati (es. Th60deg)
        continue
    x_cm = round(x_mm/10.0)

    rf = uproot.open(fpath)
    df = rf["Fotoni"].arrays(["EventID","Arrival_Time_ns","PMT_ID","E_Hit_eV"], library="pd")
    df = df[df.PMT_ID.isin([CH_A, CH_B])]
    if df.empty:
        continue
    event_ids = np.sort(df.EventID.unique())
    if EVENTS_PER_FILE is not None:
        event_ids = event_ids[:EVENTS_PER_FILE]

    print(f" -> X={x_cm:+.0f} cm | {len(event_ids)} eventi", end="")

    for gg in GAIN_LIST:
        qa_qb_pairs[float(gg)].setdefault(x_cm, [])
        ped_a[float(gg)].setdefault(x_cm, [])
        ped_b[float(gg)].setdefault(x_cm, [])

    for ev_id in event_ids:
        hits_ev = df[df.EventID == ev_id]
        # --- parte stocastica INDIPENDENTE dal guadagno: fatta una sola volta per evento ---
        pulses_ch = {}
        for pmt_id in (CH_A, CH_B):
            hits = hits_ev[hits_ev.PMT_ID == pmt_id]
            if hits.empty:
                pulses_ch[pmt_id] = None
                continue
            wl_nm = 1240.0 / hits.E_Hit_eV.values
            accepted = rng.random(len(wl_nm)) < get_quantum_efficiency(wl_nm)
            t_acc = hits.Arrival_Time_ns.values[accepted]
            if len(t_acc) == 0:
                pulses_ch[pmt_id] = None
                continue
            jitters = rng.normal(0.0, TTS_SIGMA_NS, len(t_acc))
            gfluct  = np.maximum(0.0, rng.normal(1.0, SPE_RES, len(t_acc)))
            t_s = t_acc + TRIGGER_OFFSET_NS + jitters
            dt  = time_axis[:, None] - t_s[None, :]
            shape = np.exp(-0.5*(dt/sigma_total)**2)         # (N_samp, N_ph)
            pulses_ch[pmt_id] = shape @ gfluct                # (N_samp,) unita' = 1 mV/gain-di-riferimento

        # --- per ogni guadagno: scala ampiezza, quantizza, estrai carica + piedistallo ---
        for gg in GAIN_LIST:
            amp = amp_spe_mv(gg)
            qs, peds = {}, {}
            for pmt_id in (CH_A, CH_B):
                base = pulses_ch[pmt_id]
                voltage = np.zeros(len(time_axis)) if base is None else amp * base
                wf_adc = mv_to_adc(voltage, rng)
                qs[pmt_id], peds[pmt_id] = extract_charge_and_ped(wf_adc)

            qA, qB = qs[CH_A], qs[CH_B]
            if np.isfinite(qA):
                ped_a[float(gg)][x_cm].append(peds[CH_A])
            if np.isfinite(qB):
                ped_b[float(gg)][x_cm].append(peds[CH_B])
            if np.isfinite(qA) and np.isfinite(qB) and qA > 0 and qB > 0:
                qa_qb_pairs[float(gg)][x_cm].append((qA, qB))

    print("  OK")

print("\nDigitizzazione multi-guadagno completata.")

In [ ]:
# ============================================================
# PLOT: log(Q_A/Q_B) vs X, una curva per guadagno
# Selezione IDENTICA a "Charge Log-Ratio vs X" in WC_analisys_tool:
#  - coincidenza A&B con Q>0 (gia' applicata in fase di raccolta)
#  - Q_A > mu_pedA + 3 sigma_pedA  E  Q_B > mu_pedB + 3 sigma_pedB (se soglie definite
#    e restano >=2 eventi dopo il taglio; altrimenti nessun taglio pedestal, come nello
#    script originale)
# In piu': fit lineare di ln(Q_A/Q_B) vs X e lambda = -2/slope (stessa formula usata in
# WC_analisys_tool) -> lunghezza di attenuazione effettiva della fibra, per guadagno.
# ============================================================
cmap = plt.cm.viridis(np.linspace(0, 1, len(GAIN_LIST)))

fig, ax = plt.subplots(figsize=(9.5, 6.5))
print(f"{'Guadagno':>10} | {'slope (per cm)':>15} | {'lambda (cm)':>14}")
for gg, col in zip(GAIN_LIST, cmap):
    xs_all = sorted(qa_qb_pairs[float(gg)].keys())
    xs, means, sems = [], [], []
    for xc in xs_all:
        pairs = qa_qb_pairs[float(gg)][xc]
        if len(pairs) < 2:
            continue
        qa_arr = np.array([p[0] for p in pairs])
        qb_arr = np.array([p[1] for p in pairs])

        _, _, thrA = ped_fit_thr(ped_a[float(gg)][xc])
        _, _, thrB = ped_fit_thr(ped_b[float(gg)][xc])
        if np.isfinite(thrA) and np.isfinite(thrB):
            qmask = (qa_arr > thrA) & (qb_arr > thrB)
            if qmask.sum() >= 2:
                qa_arr, qb_arr = qa_arr[qmask], qb_arr[qmask]

        if len(qa_arr) < 2:
            continue
        log_ratio = np.log(qa_arr/qb_arr)
        xs.append(xc); means.append(np.mean(log_ratio)); sems.append(np.std(log_ratio)/np.sqrt(len(log_ratio)))
    if not xs:
        continue

    lam_str = ""
    if len(np.unique(xs)) > 2:
        p_q, V_q = np.polyfit(xs, means, 1, cov=True)
        slope_q, _ = p_q
        err_l = 2*np.sqrt(np.diag(V_q))[0] / (slope_q**2)
        lam = -2.0/slope_q
        lam_str = f"  [$\\lambda$={lam:.0f}$\\pm${abs(err_l):.0f} cm]"
        print(f"{gg:>10.1e} | {slope_q:>15.5f} | {lam:>10.1f} +/- {abs(err_l):.1f}")

    ax.errorbar(xs, means, yerr=sems, marker='o', ms=5, capsize=3, lw=1.6,
                color=col, label=f"G = {gg:.1e}{lam_str}")

ax.axhline(0, color='gray', ls='--', lw=1)
ax.set_xlabel("X (cm)"); ax.set_ylabel("ln(Q_A / Q_B)")
ax.set_title(f"ln(Q_A/Q_B) vs X — scan guadagno PMT ({GEO_SEL}, {SRC_SEL})\nselezione: A&B coinc. + soglia piedistallo 3$\\sigma$ (come WC_analisys_tool)")
ax.grid(True, alpha=0.3); ax.legend(fontsize=8.5, title="Guadagno  (fit: lambda = -2/slope)")
fig.tight_layout()
out_pdf = os.path.join(OUT_DIR, f"logQ_vs_X_gain_scan_{GEO_SEL}_{SRC_SEL}.pdf")
fig.savefig(out_pdf, bbox_inches="tight")
print("salvato:", out_pdf)
plt.show()